### ELASTIC_SEARCH


In [63]:
from elasticsearch import Elasticsearch

# -----------------------
# Connect to ES 8.x server
# -----------------------
es_client = Elasticsearch(
    "http://localhost:9200",
    request_timeout=30
)

In [64]:
from elasticsearch import Elasticsearch
from elasticsearch.exceptions import NotFoundError


# -----------------------
# Define mappings
# -----------------------

# For collectionNew and collectionOld (normal text chunks)
index_settings_chunks = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "doc_id": {"type": "keyword"},
            "text": {"type": "text"},
            "metadata": {
                "properties": {
                    "chapter": {"type": "text"},
                    "section_number": {"type": "keyword"},
                    "section_title": {"type": "text"},
                    "law_name": {"type": "keyword"}
                }
            }
        }
    }
}

# For collectionMap (mapping between laws)
index_settings_map = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "source_file": {"type": "keyword"},
            "chunk_id": {"type": "keyword"},
            "fields": {
                "properties": {
                    "New_Law_Section": {"type": "keyword"},
                    "Old_Law_Section": {"type": "keyword"},
                    "Subject": {"type": "text"},
                    "Summary_of_comparison": {"type": "text"}
                }
            }
        }
    }
}

# -----------------------
# Create (or recreate) indices
# -----------------------
collections = {
    "collection-new": index_settings_chunks,
    "collection-old": index_settings_chunks,
    "collection-map": index_settings_map
}

for idx, settings in collections.items():
    # Delete if exists
    try:
        es_client.indices.delete(index=idx)
        print(f"🗑️ Deleted existing index '{idx}'.")
    except NotFoundError:
        print(f"ℹ️ Index '{idx}' not found, creating fresh.")

    # Create new
    es_client.indices.create(
        index=idx,
        settings=settings["settings"],
        mappings=settings["mappings"]
    )
    print(f"🚀 Created index '{idx}' successfully!")


🗑️ Deleted existing index 'collection-new'.
🚀 Created index 'collection-new' successfully!
🗑️ Deleted existing index 'collection-old'.
🚀 Created index 'collection-old' successfully!
🗑️ Deleted existing index 'collection-map'.
🚀 Created index 'collection-map' successfully!


In [65]:
from tqdm.auto import tqdm

In [66]:
import json
import os
collection_files = {
    "collection-new": ["bns.json", "bnss.json", "bsa.json"],
    "collection-old": ["ipc.json", "crpc.json", "iea.json"],
    "collection-map": ["mapping_of_laws.json"]
}

data_dir = "../data"

# -----------------------
# Load and index documents
# -----------------------
for collection, files in collection_files.items():
    for fname in files:
        file_path = os.path.join(data_dir, fname)
        with open(file_path, "r", encoding="utf-8") as f:
            documents = json.load(f)

        for doc in tqdm(documents, desc=f"Indexing {fname} into {collection}"):
            es_client.index(index=collection, document=doc,timeout="300s")

Indexing iea.json into collection-old: 100%|██████████| 467/467 [00:00<00:00, 997.74it/s] 
Indexing mapping_of_laws.json into collection-map: 100%|██████████| 1237/1237 [00:01<00:00, 1103.44it/s]


In [67]:
def elastic_search(query):
    """
    Runs the same search query on:
      - collection-new
      - collection-old
      - collection-map
    Returns combined results from all three.
    """

    # --- Search 1: collection-new ---
    search_new = {
        "size": 10,
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query,
                        "fields": [
                            "text^3",
                            "metadata.section_title",
                            "metadata.section_number",
                            "metadata.law_name"
                        ],
                        "type": "best_fields"
                    }
                }]
            }
        }
    }
    resp_new = es_client.search(index="collection-new", query=search_new["query"], size=search_new["size"])
    results_new = [hit["_source"] for hit in resp_new["hits"]["hits"]]

    # --- Search 2: collection-old ---
    search_old = {
        "size": 5,
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query,
                        "fields": [
                            "text^3",
                            "metadata.section_title",
                            "metadata.section_number",
                            "metadata.law_name"
                        ],
                        "type": "best_fields"
                    }
                }]
            }
        }
    }
    resp_old = es_client.search(index="collection-old", query=search_old["query"], size=search_old["size"])
    results_old = [hit["_source"] for hit in resp_old["hits"]["hits"]]

    search_map = {
        "size": 5,
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query,
                        "fields": [
                            "fields.New_Law_Section^3",
                            "fields.Old_Law_Section^3",
                            "fields.Subject",
                            "fields.Summary_of_comparison"
                        ],
                        "type": "best_fields"
                    }
                }]
            }
        }
    }
    resp_map = es_client.search(index="collection-map", query=search_map["query"], size=search_map["size"])
    results_map = [hit["_source"] for hit in resp_map["hits"]["hits"]]

    # --- Combined results ---
    combined_results = {
        "collection-new": results_new,
        "collection-old": results_old,
        "collection-map": results_map  # key updated here
    }

    return combined_results

### VECTOR_SEARCH

In [68]:
from qdrant_client import QdrantClient, models

# Initialize Qdrant client
qd_client = QdrantClient("http://localhost:6333")

In [69]:
from sentence_transformers import SentenceTransformer

# Initialize
model_handle = SentenceTransformer("BAAI/bge-large-en-v1.5")


In [70]:
collections = {
    "collection-new": "collection-new",
    "collection-old": "collection-old",
    "collection-map": "collection-map"
}


In [71]:
# Collections to keep
collections_to_keep = {
}

# Get all collections in Qdrant
all_collections = [c.name for c in qd_client.get_collections().collections]

for col in all_collections:
    if col not in collections_to_keep:
        print(f"Deleting collection: {col}...")
        qd_client.delete_collection(col)
    else:
        print(f"Keeping collection: {col}")


Deleting collection: collection-new...


In [72]:
BATCH_SIZE = 32       # Number of documents to embed and upsert at once
VECTOR_SIZE = 1024


In [73]:
import os
import json
import uuid
from tqdm import tqdm
from qdrant_client import QdrantClient, models

# -----------------------
DATA_DIR = "../data"   # Correct path to your JSON files



# -----------------------
def ensure_collection_fresh(qd_client, collection_name, vector_size=VECTOR_SIZE):
    if qd_client.collection_exists(collection_name):
        print(f"⚠️ Collection '{collection_name}' exists. Deleting it...")
        qd_client.delete_collection(collection_name)
    print(f"⚙️ Creating collection: {collection_name}")
    qd_client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )

# -----------------------
# Collections & files
# -----------------------
collection_files = {
    "collection-new": ["bns.json", "bnss.json", "bsa.json"],
    "collection-old": ["ipc.json", "crpc.json", "iea.json"],
    "collection-map": ["mapping_of_laws.json"]
}

# -----------------------
# Load & upsert into Qdrant
# -----------------------
for collection, files in collection_files.items():
    ensure_collection_fresh(qd_client, collection)

    for fname in files:
        file_path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(file_path):
            print(f"⚠️ File not found: {file_path}")
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            documents = json.load(f)

        for start in tqdm(range(0, len(documents), BATCH_SIZE), desc=f"Indexing {fname}"):
            batch_docs = documents[start:start + BATCH_SIZE]

            # Prepare texts for embedding
            texts = [
                doc.get("text", "") if collection in ["collection-new", "collection-old"]
                else json.dumps(doc, ensure_ascii=False)
                for doc in batch_docs
            ]

            # Generate embeddings using your existing model_handle
            vectors = list(model_handle.encode(texts))
            assert len(vectors) == len(batch_docs)

            points = []
            for doc, vector in zip(batch_docs, vectors):
                point_id = str(uuid.uuid4())

                # Payload differs for collection-map
                if collection in ["collection-new", "collection-old"]:
                    payload = {
                        "chunk_id": doc.get("chunk_id"),
                        "doc_id": doc.get("doc_id"),
                        "content": doc.get("text", ""),
                        "metadata": {
                            "section_number": doc.get("metadata", {}).get("section_number"),
                            "section_title": doc.get("metadata", {}).get("section_title"),
                            "law_name": doc.get("metadata", {}).get("law_name")
                        }
                    }
                else:  # collection-map
                    payload = {
                        "chunk_id": doc.get("chunk_id"),
                        "source_file": doc.get("source_file"),
                        "fields": doc.get("fields", {})
                    }

                points.append(models.PointStruct(id=point_id, vector=vector, payload=payload))

            # Upsert batch
            qd_client.upsert(collection_name=collection, points=points)


⚙️ Creating collection: collection-new


Indexing bsa.json: 100%|██████████| 6/6 [01:23<00:00, 13.86s/it]


⚙️ Creating collection: collection-old


Indexing iea.json: 100%|██████████| 15/15 [01:44<00:00,  7.00s/it]


⚙️ Creating collection: collection-map


Indexing mapping_of_laws.json: 100%|██████████| 39/39 [03:29<00:00,  5.38s/it]


In [74]:
def vector_search(question):
    if not question:
        question = " "  # fallback for empty query

    # Generate embedding
    vector = list(model_handle.encode([question]))[0]

    # Define which collections to search and limits
    search_config = [
        ("collection-new", 10),
        ("collection-old", 5),
        ("collection-map", 5)
    ]

    results = {}

    for col_name, limit in search_config:
        # ✅ Using query_points (returns object with .points)
        query_points = qd_client.query_points(
            collection_name=col_name,
            query=vector,
            limit=limit,
            with_payload=True
        )

        cleaned_results = []

        for p in query_points.points:
            payload = p.payload or {}

            if col_name == "collection-map":
                fields = payload.get("fields", {})
                cleaned_results.append({
                    "chunk_id": payload.get("chunk_id"),
                    "source_file": payload.get("source_file"),
                    "New_Law_Section": fields.get("New_Law_Section"),
                    "Old_Law_Section": fields.get("Old_Law_Section"),
                    "Subject": fields.get("Subject"),
                    "Summary_of_comparison": fields.get("Summary_of_comparison"),
                })
            else:
                meta = payload.get("metadata", {}) or {}
                cleaned_results.append({
                    "chunk_id": payload.get("chunk_id"),
                    "doc_id": payload.get("doc_id"),
                    "text": payload.get("content"),
                    "section_number": meta.get("section_number"),
                    "section_title": meta.get("section_title"),
                    "law_name": meta.get("law_name"),
                })

        results[col_name] = cleaned_results

    return results


## HYBRID SEARCH


In [75]:
import numpy as np

def hybrid_search(question, alpha=0.3):
    """
    Hybrid search:
    1. Get chunks from vector_search
    2. Fetch embeddings from Qdrant for each chunk
    3. Combine with ElasticSearch scores
    4. Compute hybrid_score = alpha*_score + (1-alpha)*vector_score
    5. Remove duplicates and return top-k per collection
    """

    if not question:
        question = " "

    # --- Step 1: Generate query embedding ---
    query_vector = list(model_handle.encode([question]))[0]

    # --- Step 2: Get ElasticSearch results ---
    es_results = elastic_search(question)

    # --- Step 3: Get chunks from vector_search (without embeddings) ---
    vector_chunks = vector_search(question)

    # --- Step 4: Collection config (top-k per collection) ---
    search_config = [
        ("collection-new", 10),
        ("collection-old", 5),
        ("collection-map", 5)
    ]

    hybrid_results = {}

    for collection, top_k in search_config:
        combined = []
        seen_ids = set()

        # --- Add ElasticSearch results ---
        for r in es_results.get(collection, []):
            chunk_id = r.get("chunk_id") or r.get("doc_id")
            if chunk_id in seen_ids:
                continue
            seen_ids.add(chunk_id)
            combined.append({
                **r,
                "hybrid_score": alpha * r.get("_score", 1.0)
            })

        # --- Fetch embeddings from Qdrant for each chunk ---
        for r in vector_chunks.get(collection, []):
            chunk_id = r.get("chunk_id") or r.get("doc_id")
            if chunk_id in seen_ids:
                continue
            seen_ids.add(chunk_id)

            # Fetch embedding from Qdrant
            try:
                point = qd_client.retrieve(
                    collection_name=collection,
                    ids=[chunk_id],
                    with_vector=True
                )
                stored_vector = point[0].vector if point else None
            except Exception:
                stored_vector = None

            # Compute cosine similarity
            vector_score = 0.0
            if stored_vector:
                v1 = np.array(query_vector)
                v2 = np.array(stored_vector)
                vector_score = float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8))

            combined.append({
                **r,
                "hybrid_score": (1 - alpha) * vector_score
            })

        # --- Sort and keep top-k ---
        combined.sort(key=lambda x: x['hybrid_score'], reverse=True)
        hybrid_results[collection] = combined[:top_k]

    return hybrid_results


#### Save questions to csv file

In [76]:
import pandas as pd
import json
import os
from tqdm import tqdm

# ----------------------------------------------------
# 1. Define the file paths
# ----------------------------------------------------
file_names = [
    "../data/questions/llm_questions_mapping.json",
    "../data/questions/llm_questions_newlaws.json",
    "../data/questions/llm_questions_oldlaws.json"
]

# ----------------------------------------------------
# 2. Helper function to infer collection from filename
# ----------------------------------------------------
def infer_collection_from_filename(file_path):
    file_name = os.path.basename(file_path).lower()
    if "mapping" in file_name:
        return "collection-map"
    elif "newlaws" in file_name:
        return "collection-new"
    elif "oldlaws" in file_name:
        return "collection-old"
    else:
        return "unknown_collection"

# ----------------------------------------------------
# 3. Function to load questions from a JSON file
# ----------------------------------------------------
def load_questions_with_chunk_id_as_q_id(file_path):
    all_records = []
    collection = infer_collection_from_filename(file_path)

    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found at: {file_path}")

    for entry in data:
        q_id = entry.get("chunk_id", "NO_CHUNK_ID")
        questions = entry.get("llm_questions", [])
        for question in questions:
            all_records.append({
                "q_id": q_id,
                "question": question,
                "collection": collection
            })
    return all_records

# ----------------------------------------------------
# 4. Load and process all files
# ----------------------------------------------------
all_questions = []
for file_name in file_names:
    try:
        all_questions.extend(load_questions_with_chunk_id_as_q_id(file_name))
        print(f"✅ Successfully loaded: {file_name}")
    except FileNotFoundError:
        print(f"❌ ERROR: File not found: {file_name}. Please check the path.")
    except Exception as e:
        print(f"❌ An error occurred processing {file_name}: {e}")

# ----------------------------------------------------
# 5. Create a Pandas DataFrame
# ----------------------------------------------------
if all_questions:
    df = pd.DataFrame(all_questions)

    print("\n" + "=" * 40)
    print("DataFrame Created Successfully:")
    print("=" * 40)
    print(df.head(5))
    print("-" * 40)
    print(f"Total rows in DataFrame: {len(df)}")
    print(f"Number of unique chunk_id/q_id values: {df['q_id'].nunique()}")
    print(f"Collections present: {df['collection'].unique()}")
else:
    print("\nDataFrame creation failed: No data was loaded.")

# ----------------------------------------------------
# 6. Optional: save combined data to CSV
# ----------------------------------------------------
output_csv_path = "../data/all-questions.csv"
df.to_csv(output_csv_path, index=False)
print(f"\n✅ Combined questions saved to {output_csv_path}")


✅ Successfully loaded: ../data/questions/llm_questions_mapping.json
✅ Successfully loaded: ../data/questions/llm_questions_newlaws.json
✅ Successfully loaded: ../data/questions/llm_questions_oldlaws.json

DataFrame Created Successfully:
               q_id                                           question  \
0  BNSS_to_CrPC_268  What does the chunk_id BNSS_to_CrPC_268 imply ...   
1  BNSS_to_CrPC_268  In the absence of textual content, what type o...   
2  BNSS_to_CrPC_268  If you were to populate this chunk with CrPC s...   
3  BNSS_to_CrPC_268  How would you annotate this chunk to indicate ...   
4  BNSS_to_CrPC_268  What metadata fields would be necessary to loc...   

       collection  
0  collection-map  
1  collection-map  
2  collection-map  
3  collection-map  
4  collection-map  
----------------------------------------
Total rows in DataFrame: 6700
Number of unique chunk_id/q_id values: 1279
Collections present: ['collection-map' 'collection-new' 'collection-old']

✅ Combin

In [77]:
import os

# Make sure the directory exists
os.makedirs("../data", exist_ok=True)

# Save the DataFrame
df.to_csv("../data/all-questions.csv", index=False)
print("Saved to ../data/all-questions.csv")



Saved to ../data/all-questions.csv


In [78]:
import pandas as pd

df_allquestions = pd.read_csv('../data/all-questions.csv')
all_questions = df_allquestions.to_dict(orient='records')
all_questions

[{'q_id': 'BNSS_to_CrPC_268',
  'question': 'What does the chunk_id BNSS_to_CrPC_268 imply about the source and target statutes involved?',
  'collection': 'collection-map'},
 {'q_id': 'BNSS_to_CrPC_268',
  'question': 'In the absence of textual content, what type of questions would be most appropriate to evaluate the BNSS_to_CrPC_268 mapping?',
  'collection': 'collection-map'},
 {'q_id': 'BNSS_to_CrPC_268',
  'question': 'If you were to populate this chunk with CrPC section 268 content, what would be the most likely legal issue area to focus on?',
  'collection': 'collection-map'},
 {'q_id': 'BNSS_to_CrPC_268',
  'question': 'How would you annotate this chunk to indicate its empty text and its purpose within a larger dataset?',
  'collection': 'collection-map'},
 {'q_id': 'BNSS_to_CrPC_268',
  'question': 'What metadata fields would be necessary to locate the actual legal text corresponding to BNSS_to_CrPC_268 in a repository?',
  'collection': 'collection-map'},
 {'q_id': 'BSA_to_IE

## Evaluation Results

In [79]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)


In [80]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break  # stop after the first relevant item
    return total_score / len(relevance_total)


In [81]:

def evaluate(all_questions, search_function):
    relevance_total = []

    for q in tqdm(all_questions):
        q_id = q['q_id']
        collection_origin = q['collection']

        # Run the search using the passed function
        results = search_function(q)
        
        # Extract the relevant collection results
        chunks = results[collection_origin]
        
        # Compute relevance for this collection
        relevance = [d['chunk_id'] == q_id for d in chunks]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }


#### ElasticSearch Evaluation Results

In [82]:
evaluate(all_questions, lambda q: elastic_search(q['question']))

100%|██████████| 6700/6700 [01:01<00:00, 108.75it/s]


{'hit_rate': 0.6080597014925373, 'mrr': 0.506540274816395}

#### VectorSearch Evaluation Results

In [83]:
evaluate(all_questions, lambda q: vector_search(q['question']))

100%|██████████| 6700/6700 [50:14<00:00,  2.22it/s]  


{'hit_rate': 0.6094029850746269, 'mrr': 0.5004218194740593}

In [84]:
evaluate(all_questions, lambda q: hybrid_search(q['question'], alpha= 0.3))

100%|██████████| 6700/6700 [51:13<00:00,  2.18it/s]  


{'hit_rate': 0.6134328358208955, 'mrr': 0.508428334517887}